# ⚽ Football Vision — Analisi Tattica su GPU (Colab)

Tutto su **GPU gratuita**: codice + modelli, una clip, e in output
**radar 2D + heatmap + dashboard + report scouting (metriche atletiche, tattiche, performance)**.

## Come si usa
1. **Runtime → Cambia tipo di runtime → GPU (T4)** → Salva.
2. Esegui le celle in ordine (`Shift+Invio`).
3. Per la clip: usa **una sola** delle opzioni (Carica file consigliata).
4. Lancia l'analisi e guarda i risultati.

## 1) Verifica GPU

In [ ]:
import torch
print('✅ GPU attiva:', torch.cuda.get_device_name(0)) if torch.cuda.is_available() else print('❌ GPU NON attiva! Runtime → Cambia tipo di runtime → GPU (T4).')

## 2) Scarica codice + modelli + librerie (~1-2 min)

In [ ]:
!pip -q install ultralytics supervision scikit-learn 2>/dev/null
import os, shutil
if os.path.exists('football-vision'):
    shutil.rmtree('football-vision')
!git clone -q https://github.com/sebavidal2001/football-vision.git
%cd football-vision
import urllib.request
os.makedirs('vista_tattica', exist_ok=True)
os.makedirs('clips_input', exist_ok=True)
modelli = {
    'vista_tattica/yolo-football-pitch-detection.pt':
        'https://huggingface.co/martinjolif/yolo-football-pitch-detection/resolve/main/yolo-football-pitch-detection.pt',
    'vista_tattica/giocatori_calcio.pt':
        'https://huggingface.co/uisikdag/yolo-v8-football-players-detection/resolve/main/best.pt',
}
for dst, url in modelli.items():
    if not os.path.exists(dst):
        print('Scarico', os.path.basename(dst), '...'); urllib.request.urlretrieve(url, dst)
print('\n✅ Tutto pronto.')

## 3) Scegli la clip — usa UNA sola opzione

### ✅ OPZIONE A (consigliata): carica un file dal tuo PC
Ritaglia prima una clip (1-5 min) col programma `AVVIA_ritaglia.bat`, poi caricala qui.

In [ ]:
from google.colab import files
import os
os.makedirs('clips_input', exist_ok=True)
up = files.upload()                       # scegli la clip dal PC
src = list(up.keys())[0]
!ffmpeg -y -i "{src}" -c:v libx264 -pix_fmt yuv420p -an clips_input/clip_input.mp4 2>/dev/null
print('\n✅ Caricata e normalizzata.')

### OPZIONE B: da Google Drive (per PARTITE INTERE / file grandi)
Carica il video sul tuo Google Drive, poi monta il Drive e indica il percorso.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# PERCORSO = '/content/drive/MyDrive/partita.mp4'   # <-- il tuo file su Drive
# import os; os.makedirs('clips_input', exist_ok=True)
# !ffmpeg -y -i "{PERCORSO}" -c:v libx264 -pix_fmt yuv420p -an clips_input/clip_input.mp4 2>/dev/null
# print('✅ Pronta da Drive.')

### OPZIONE C: da YouTube (spesso BLOCCATO su Colab — 'Sign in to confirm you're not a bot')
Funziona solo passando i **cookie** del tuo browser. Se dà errore, usa l'Opzione A.

In [ ]:
# !pip -q install yt-dlp
# LINK='https://youtu.be/...'; INIZIO='00:10:00'; FINE='00:13:00'
# # serve un file cookies.txt caricato (esportato dal browser con un'estensione 'Get cookies.txt')
# from google.colab import files; files.upload()   # carica cookies.txt
# !yt-dlp --cookies cookies.txt --download-sections "*{INIZIO}-{FINE}" \
#   -f "bestvideo[height<=720][vcodec^=avc1]+bestaudio/best[height<=720]" \
#   --merge-output-format mp4 -o clips_input/clip_dl.mp4 "{LINK}"
# !ffmpeg -y -i clips_input/clip_dl.mp4 -c:v libx264 -pix_fmt yuv420p -an clips_input/clip_input.mp4 2>/dev/null
# print('OK' if os.path.exists('clips_input/clip_input.mp4') else 'ERRORE')

## 4) Analisi su GPU
Produce radar 2D, heatmap, dashboard e **report scouting**.
Clip breve: `SALTO=2`. Partita intera: `SALTO=3`, `OGNI_CAMPO=2`.

In [ ]:
SALTO = 2
OGNI_CAMPO = 1

import os
video = 'clips_input/clip_input.mp4'
base = os.path.splitext(os.path.basename(video))[0]
csv_pos = f'output/POSIZIONIAUTO_{base}.csv'

print('▶ 1/4 Radar 2D + rilevamento giocatori...')
!python vista_tattica/genera_radar_auto.py "{video}" --salto {SALTO} --ogni_campo {OGNI_CAMPO} --imgsz 1280
print('\n▶ 2/4 Heatmap + statistiche...')
!python analisi/stats_giocatori.py "{csv_pos}" --min_rilevazioni 20
print('\n▶ 3/4 Dashboard di confronto...')
!python analisi/confronto_giocatori.py "output/STATISTICHE_{base}.csv"
print('\n▶ 4/4 Metriche scouting (atletiche + tattiche + performance)...')
!python analisi/metriche_avanzate.py "{csv_pos}" --min_rilevazioni 20
print('\n✅ FATTO.')

## 5) Risultati

In [ ]:
from IPython.display import Image, display
import glob
for pattern, titolo in [('output/REPORT_*.png', '=== REPORT SCOUTING ==='),
                        ('output/DASHBOARD_*.png', '=== DASHBOARD CONFRONTO ===')]:
    for f in glob.glob(pattern):
        print(titolo); display(Image(f))
for f in sorted(glob.glob('output/heatmaps_*/_SQUADRA_*.png')):
    display(Image(f, width=520))

In [ ]:
import glob, pandas as pd
mg = glob.glob('output/METRICHE_GIOCATORI_*.csv')
if mg:
    display(pd.read_csv(mg[0]))

In [ ]:
import glob
rad = glob.glob('output/RADARAUTO_*.mp4')
if rad:
    !ffmpeg -y -i "{rad[0]}" -vcodec libx264 -pix_fmt yuv420p output/_radar_web.mp4 2>/dev/null
    from IPython.display import Video, display
    display(Video('output/_radar_web.mp4', embed=True, width=900))

## 6) Scarica tutti i risultati (zip)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('risultati', 'zip', 'output')
files.download('risultati.zip')